In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

In [6]:
host_trust = pd.read_csv("../Data/HostTrust_Output.csv")[
    ["listing_id", "HostTrust", "HostTrust_Components_Used", "review_scores_rating"]
]
listing_iq = pd.read_csv("../Data/ListingIQ_Output.csv")[
    ["listing_id", "ListingIQ", "ListingIQ_Components_Used"]
]
experience_iq = pd.read_csv("../Data/ExperienceIQ_Output.csv")[
    ["listing_id", "ExperienceIQ"]
]

master = (
    host_trust
    .merge(listing_iq, on="listing_id", how="left")
    .merge(experience_iq, on="listing_id", how="left")
)

print(master.shape)
master.head()

(279434, 7)


,listing_id,HostTrust,HostTrust_Components_Used,review_scores_rating,ListingIQ,ListingIQ_Components_Used,ExperienceIQ
0,281420,45.477167,3,100.0,21.812885,5.0,100.0
1,3705183,42.189042,3,100.0,22.830290,5.0,100.0
2,4082273,32.385460,3,100.0,22.152020,5.0,100.0
3,4797344,41.946353,3,100.0,21.812885,5.0,100.0
4,4823489,30.551812,3,100.0,24.186830,5.0,100.0


In [7]:
master.info()
master.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279434 entries, 0 to 279433
Data columns (total 7 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   listing_id                 279434 non-null  int64  
 1   HostTrust                  279434 non-null  float64
 2   HostTrust_Components_Used  279434 non-null  int64  
 3   review_scores_rating       188199 non-null  float64
 4   ListingIQ                  279433 non-null  float64
 5   ListingIQ_Components_Used  279433 non-null  float64
 6   ExperienceIQ               187812 non-null  float64
dtypes: float64(5), int64(2)
memory usage: 14.9 MB


,listing_id,HostTrust,HostTrust_Components_Used,review_scores_rating,ListingIQ,ListingIQ_Components_Used,ExperienceIQ
count,2.794340e+05,279434.000000,279434.000000,188199.000000,279433.000000,279433.000000,187812.000000
mean,2.637677e+07,29.080200,4.676149,93.405395,29.379194,4.895009,93.356013
std,1.442383e+07,14.156633,1.370729,10.070857,6.360351,0.306543,11.473643
min,2.577000e+03,0.013483,3.000000,20.000000,2.632232,4.000000,0.000000
25%,1.384404e+07,17.985930,3.000000,91.000000,24.710910,5.000000,91.307500
50%,2.766007e+07,28.328820,5.000000,96.000000,28.672754,5.000000,100.000000
75%,3.977787e+07,38.272440,6.000000,100.000000,33.004341,5.000000,100.000000
max,4.834353e+07,95.955188,6.000000,100.000000,79.359506,5.000000,100.000000


In [8]:
master["KPIs_Available"] = (
    master[
        ["HostTrust","ListingIQ","ExperienceIQ"]
    ]
    .notna()
    .sum(axis=1)
)

master["KPIs_Available"].value_counts().sort_index()

KPIs_Available
1         1
2     91621
3    187812
Name: count, dtype: int64

In [9]:
master["KPI_Combination"] = (
    master[
        ["HostTrust","ListingIQ","ExperienceIQ"]
    ]
    .notna()
    .astype(int)
    .astype(str)
    .agg("".join, axis=1)
)

master["KPI_Combination"].value_counts()

KPI_Combination
111    187812
110     91621
100         1
Name: count, dtype: int64

In [10]:
master[
    ["HostTrust", "ListingIQ", "ExperienceIQ"]
].corr()

,HostTrust,ListingIQ,ExperienceIQ
HostTrust,1.000000,0.095753,0.195373
ListingIQ,0.095753,1.000000,0.086177
ExperienceIQ,0.195373,0.086177,1.000000
